# Recipe Retrieval - An Information Retrieval System
This repository defines an Information Retrieval System that can be used for recipe datasets. Given a phrase query, it will return the recipes that are more relevant in the current dataset, which contains approximately 60 thousands documents.

The recipes, in `.json` format, are pre-processed and then saved as matrixes of **TF-TDF** values, coherently with the Vector Space Model. The words are normalized and saved as an **inverted index**: each term is mapped to a posting list, which contains the document-ids of the recipes containing that term.

The inverted index is built in such a way that each term is formed by the couple `word` and `zone`. This choice was made in order to give much more importance in the retrieval to words contained in titles with respect to the ones in the ingredients and instructions parts.

For the actual retrieval, the `search engine` transforms the queries into vectors and calculates the cosine similarity with the documents vectors. The ones with larger cosine similarity are returned, ordered. The user can also provide a **relevance feedback**, that will transpose the query in the vector space closer to the relevant documents, using the Rocchio algorithm.

To make these calculations efficient, given the large number of documents and terms, the `scipy` sparce arrays were used. This made the cosine similarity calculation trivial and lowered significantly the amount of space needed to save the matrix, because of the large number of 0-values in the vectors.

A test benchmark was also developed to test our system on a subset of recipes, based on automatically generated queries. The system gave strong results and proved its robustness even with original phrase queries.

Developed by: Gabriele Pasqualini, Federico Marenco and Anna Guccione.


## Project Structure

```text
recipe-retrieval/
├── data/                       
│   ├── corpus/                 # * corpus of recipes
│   └── bin/                    # * .pkl of the inverted index and document vectors
│   └── benchmark/              # * data used for the benchmark evaluation
├── src/                        
│   ├── core/                  
│   │   ├── models.py           # Term, Posting, PostingList classes
│   │   └── index.py            # InvertedIndex class
│   ├── search/                 
│   │   ├── engine.py           # SearchEngine class
│   │   └── query.py            # retrieval logic
│   ├── utils/                  
│   │   ├── tokenizer.py        # tokenizing logic
│   │   ├── build_index.py      # index construction logic
│   │   └── corpus_parser.py    # corpus preprocessing logic
│   ├── benchmark/               
│   │   └── benchmark.py        # evaluation of the system
│   └── web/                    # integration with Flask
├── config.yaml                 # parameters
├── requirements.txt
└── README.md
```

## Steps

In [1]:
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2

import yaml

with open("config.yaml", "r") as f:
        config = yaml.safe_load(f)

  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached symspellpy-6.9.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached regex-2026.4.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached editdistpy-0.2.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.meta

### Corpus

#### Downloading the corpus

This method scrapes from an online source the dataset that was used for the Inverted Index. It dowloads 3 json files and keeps just `RAW JSON`.

In [2]:
import requests, zipfile, io, os, shutil
RAW_JSON = "recipes_raw_nosource_fn.json"

def download_corpus(url, dir, raw_path):
    
    print("Downloading the file...")
    
    raw_dir = os.path.join(dir, "raw")
    r = requests.get(url)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall(raw_dir)
    for file in os.listdir(raw_dir):
        filename = os.fsdecode(file)
        if filename != RAW_JSON:
            os.remove(os.path.join(raw_dir, filename))
    
    source_file = os.path.join(raw_dir, RAW_JSON)
    
    shutil.move(str(source_file), str(raw_path))
    os.rmdir(raw_dir)
    
    print(f"Download completed: saved in {str(raw_path)}")

In [3]:
URL = "https://eightportions.com/recipes_raw.zip"
    
DIR = config["corpus"]["dir"]
RAW_PATH = config["corpus"]["raw"]

download_corpus(URL, DIR, RAW_PATH)

Download completed: saved in data/corpus/recipes_raw.json


#### Preprocessing the corpus
Converts a raw recipe dataset into a clean indexed corpus.
Skips recipes with missing title, ingredients, or instructions. If update=True, merges new recipes into an existing corpus file instead of rebuilding from scratch.

In [4]:
import json

def create_recipe_corpus(source_path, destination_path, update=False):

    with open(source_path, "r") as source_file:
        data = json.load(source_file)

    # If updating, start from the existing corpus so previous entries are preserved
    if update:
        with open(destination_path, "r") as corpus_file:
            recipe_corpus = json.load(corpus_file)
    else:
        recipe_corpus = {}

    doc_id = 0
    for index in data:
        recipe_json = data[index]

        # Skip incomplete records — all three fields are required for indexing
        if not (recipe_json
                and recipe_json.get('ingredients')
                and recipe_json.get('instructions')
                and recipe_json.get('title')):
            continue

        recipe_corpus[doc_id] = {
            'title':        recipe_json.get('title'),
            'ingredients':  "\n".join(recipe_json.get('ingredients')),
            'instructions': recipe_json.get('instructions')
        }
        doc_id += 1
    
    with open(destination_path, "w") as destination_file:
        json.dump(recipe_corpus, destination_file, indent=4)

In [5]:
CLEAN_PATH = config["corpus"]["clean"]

print("Pre-processing the corpus...")
create_recipe_corpus(RAW_PATH, CLEAN_PATH, update=False)
print(f"Pre-processing completed. Corpus available at {CLEAN_PATH}")

with open(CLEAN_PATH, "r") as source_file:
    corpus = json.load(source_file)
    
print("First recipe:", corpus[str(0)])

Pre-processing the corpus...
Pre-processing completed. Corpus available at data/corpus/recipes.json
First recipe: {'title': "Grammie Hamblet's Deviled Crab", 'ingredients': '1/2 cup celery, finely chopped\n1 small green pepper finely chopped\n1/2 cup finely sliced green onions\n1/4 cup chopped parsley\n1 pound crabmeat\n1 1/4 cups coarsely crushed cracker crumbs\n1/2 teaspoon salt\n3/4 teaspoons dry mustard\nDash hot sauce\n1/4 cup heavy cream\n1/2 cup melted butter', 'instructions': 'Toss ingredients lightly and spoon into a buttered baking dish. Top with additional crushed cracker crumbs, and brush with melted butter. Bake in a preheated at 350 degrees oven for 25 to 30 minutes or until delicately browned.'}


### Index
#### Building the index
Here a subclass of the acutal Inverted Index class is built as a toy example, with an entry cap, in order to cut execution time in the jupiter notebook.

In [6]:
from src.core.index import InvertedIndex
from tqdm import tqdm
from src.core.models import Term, PostingList
from src.utils.tokenize import tokenize

class ExampleInvertedIndex(InvertedIndex):
    
    def populate_index(self, corpus):
        # we take just the first 
        corpus = {k: corpus[str(k)] for k in range(400)}
        
        # First pass: build posting lists for every (token, zone) pair
        for doc_id in tqdm(corpus, desc="Indexing", unit="doc"):
            for zone in self.zones:
                text = corpus[doc_id][zone]
                for token in tokenize(text):
                    term = Term(token, zone)
                    if term not in self.index:
                        self.index[term] = PostingList()
                    self.index[term].add_occurrence(doc_id)

        # Second pass: compute IDF and populate vocab + spell-checker dictionary
        for position, term in enumerate(self.index):
            posting_list = self.index[term]
            term.update_idf(len(corpus), len(posting_list))
            self.vocab[term] = position
            self.spell.create_dictionary_entry(term.word, count=1)

In [7]:
print("Building Inverted Index...")
idx = ExampleInvertedIndex(corpus, zones=config['settings']['zones'])

Building Inverted Index...


Indexing:   0%|          | 0/400 [00:00<?, ?doc/s]

Indexing: 100%|██████████| 400/400 [00:06<00:00, 63.81doc/s] 


#### Vector matrices

This method creates the matrix of the document vectors, in line with the Vector Space Model. To do that, the module sparce from scipy was used, to avoid the waste of storage.

In [8]:
from scipy.sparse import csr_array
from sklearn.preprocessing import normalize

def build_doc_vectors(inverted_index, corpus):

    number_of_terms = len(inverted_index)
    number_of_docs = len(corpus)

    vocab = inverted_index.vocab

    # for the doc vectors, we generate a sparse array to avoid wasting space with zeros entries
    # this data structure instead of saving a #docs x #terms matrix, saves the value of tf-idf and the corresponding coordinates
    rows, cols, data = [], [], []

    for term, postings in inverted_index:
        for posting in postings:
            rows.append(posting.doc_id)
            cols.append(vocab[term])
            data.append(posting.tf * term.idf)

    # generate the sparse matrix
    vectors = csr_array((data, (rows, cols)), shape=(number_of_docs, number_of_terms))
    return  normalize(vectors, norm='l2', axis=1)


In [9]:
print("Building Document Vectors...")
doc_vectors = build_doc_vectors(idx, corpus)
print("Vector matrix built.")

first_recipe =  doc_vectors[0]
first_recipe = first_recipe.toarray().flatten()

print("\nFirst recipe vector:")
for i, idf in enumerate(first_recipe):
    if idf != 0:
        term = idx.get_term(i)
        print(f"{term} -> {idf}")

Building Document Vectors...
Vector matrix built.

First recipe vector:
additional.instructions -> 0.15280060701539225
bake.instructions -> 0.048989940391944736
baking.instructions -> 0.1452051176023112
brown.instructions -> 0.04333458936174081
brush.instructions -> 0.10656910732603556
butter.instructions -> 0.10067994825823745
butter.ingredients -> 0.050339974129118725
celery.ingredients -> 0.13282206433584998
chop.ingredients -> 0.12135689116479394
coarsely.ingredients -> 0.1501365447666024
crab.title -> 0.20693269399035907
crabmeat.ingredients -> 0.2159162512212242
cracker.instructions -> 0.19933720457727802
cracker.ingredients -> 0.20693269399035907
cream.ingredients -> 0.07808539126056178
crumb.instructions -> 0.15280060701539225
crumb.ingredients -> 0.1817627069257997
crush.instructions -> 0.18695415131081677
crush.ingredients -> 0.12930105615733042
cup.ingredients -> 0.04458911229634768
dash.ingredients -> 0.19275769237447685
degree.instructions -> 0.06781680445628287
delicately

## Search engine

In [44]:
query = "Peperoni Pizzaaa with the soucee"

#### 

#### Creation of the query vector:

In [45]:
import numpy as np

weights = {"title": 0.7, "instructions": 0.1, "ingredients": 0.2}
vocab = idx.vocab
vector = np.zeros(len(vocab))
for token in tokenize(query):
    print(token, end = "")
    token = idx.correct(token)
    print(f" --> {token}")
    for zone, weight in weights.items():
        term = Term(token, zone)
        if term in idx:
            vector[vocab[term]] += weight

norma = np.linalg.norm(vector)
if norma > 0:
    vector /= norma
    
vector = csr_array(vector)

peperoni --> pepperoni
pizzaaa --> pizza
soucee --> sauce


#### Computing cosine similarity
Similarities are computed with dot product between the query vector and the document vectors matrix.
The 10 highest cosine similarities documents are printed

In [49]:
similarities = vector.dot(doc_vectors.T).toarray().flatten()
top_10_indices = np.argsort(similarities)[::-1][:10]

for doc_id in top_10_indices:
    title = corpus.get(str(doc_id), {}).get('title')
    print(f"ID: {doc_id} | Sim: {similarities[doc_id]:.4f} | Recipe: {title}")

ID: 373 | Sim: 0.1157 | Recipe: Shrimp with Cocktail Sauce
ID: 81 | Sim: 0.1123 | Recipe: Supreme Pizza Burgers
ID: 384 | Sim: 0.1121 | Recipe: Bordelaise Sauce
ID: 295 | Sim: 0.0988 | Recipe: Horseradish Cream Sauce
ID: 260 | Sim: 0.0950 | Recipe: Vegetable Pizza
ID: 300 | Sim: 0.0920 | Recipe: Hot and Sweet Dipping Sauce
ID: 45 | Sim: 0.0890 | Recipe: Soppressata Pizzas
ID: 35 | Sim: 0.0871 | Recipe: Pizza with Fried Calamari (from Al Forno)
ID: 209 | Sim: 0.0854 | Recipe: Oven "Fried" Pizza
ID: 157 | Sim: 0.0825 | Recipe: Whiskey Cream Sauce


#### Relevance feedback with Rocchio algorithm

In [54]:
def rocchio_algorithm(query_vec, relevant_vecs, non_relevant_vecs, alpha=1.0, beta=0.75, gamma=0.15):

    relevant_centroid = (
        sum(relevant_vecs) / relevant_vecs.shape[0] 
        if relevant_vecs.shape[0] > 0 else 0)
    non_relevant_centroid = (
        sum(non_relevant_vecs) / non_relevant_vecs.shape[0]
        if non_relevant_vecs.shape[0] > 0 else 0
    )

    return (alpha * query_vec) + (beta * relevant_centroid) - (gamma * non_relevant_centroid)

In [55]:
# Given the query and the set of user choices for relevant recipes a new query vector is computed with Rocchio
def apply_relevance_feedback(vec_q, relevant_ids, all_top_ids):
        relevant_vecs = doc_vectors[relevant_ids]
        non_relevant_ids = [idx for idx in all_top_ids if idx not in relevant_ids]
        non_relevant_vecs = doc_vectors[non_relevant_ids]
        return rocchio_algorithm(vec_q, relevant_vecs, non_relevant_vecs)

Recomputing the cosine similarity with the new query corrected with relevance feedback

In [56]:
relevant_ids = [140, 85] # relevant ids choosen by the user

corrected_vector = apply_relevance_feedback(vector, relevant_ids, top_10_indices)
similarities = corrected_vector.dot(doc_vectors.T).toarray().flatten()
top_10_indices = np.argsort(similarities)[::-1][:10]

for doc_id in top_10_indices:
    title = corpus.get(str(doc_id), {}).get('title')
    print(f"ID: {doc_id} | Sim: {similarities[doc_id]:.4f} | Recipe: {title}")

ID: 140 | Sim: 0.3795 | Recipe: Cheese Steak Egg Rolls with Ranch Pepper Rings
ID: 85 | Sim: 0.3785 | Recipe: Turkey Sweet Potato Casserole
ID: 296 | Sim: 0.1497 | Recipe: The Crab Rangoonies
ID: 93 | Sim: 0.1439 | Recipe: Joe's "Say Cheese" Cheesecake with Fresh Strawberry Sauce
ID: 320 | Sim: 0.1300 | Recipe: Vegetable Summer Rolls
ID: 81 | Sim: 0.1296 | Recipe: Supreme Pizza Burgers
ID: 182 | Sim: 0.1165 | Recipe: Nutty Brittle
ID: 112 | Sim: 0.1097 | Recipe: Roasted Turkey with Pomegranate Sauce and Wild Rice and Goat Cheese Stuffing
ID: 260 | Sim: 0.1083 | Recipe: Vegetable Pizza
ID: 373 | Sim: 0.1041 | Recipe: Shrimp with Cocktail Sauce


## Benchmark